# PROTOTYPE — Qwen3-VL F0-R4 fallback contract qualification

**Question:** can the exact Qwen3-VL-8B-Instruct-FP8 Kaggle mount and pinned vLLM runtime cold-install without network access, become ready, accept the primary route's exact visual request under its served-model alias, emit one parsed native `take_action(LEFT)` call, and release the process group and GPU memory on shutdown?

This notebook performs one model request. It grants the fallback deployment qualification only—no Tycho behavioral authority, ARC action, score, or optimization.

In [ ]:
import base64
import json
import os
import signal
import threading
import time
from pathlib import Path

F0_RUN_STARTED = time.monotonic()
F0_HARD_DEADLINE = F0_RUN_STARTED + 1200
F0_OPERATION_DEADLINE = F0_HARD_DEADLINE - 90

def f0_hard_watchdog():
    time.sleep(1200)
    record = {
        'status': 'hard_deadline_elapsed',
        'hard_deadline_s': 1200,
        'notebook_pid': os.getpid(),
    }
    Path('/kaggle/working/qwen3-vl-f0-r4-hard-timeout.json').write_text(
        json.dumps(record, indent=2) + '\n'
    )
    process_files = [
        Path('/kaggle/working/qwen3-vl-f0-r4-outer-process.json'),
        Path('/kaggle/working/qwen3-vl-f0-r4-probe/server-process.json'),
    ]
    for process_file in process_files:
        if process_file.is_file():
            try:
                process = json.loads(process_file.read_text())
                os.killpg(int(process['pgid']), signal.SIGKILL)
            except (KeyError, ProcessLookupError, ValueError, json.JSONDecodeError):
                pass
    os.kill(os.getpid(), signal.SIGKILL)

threading.Thread(target=f0_hard_watchdog, daemon=True).start()
Path('/tmp/PROTOTYPE-qwen3_vl_f0_probe.py').write_bytes(
    base64.b64decode('...')
)


In [ ]:
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

def run_bounded(command, stdout, merge_stderr=True):
    remaining = F0_OPERATION_DEADLINE - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('F0 operation deadline elapsed before subprocess launch')
    process = subprocess.Popen(
        command,
        stdout=stdout,
        stderr=subprocess.STDOUT if merge_stderr else None,
        start_new_session=True,
    )
    try:
        returncode = process.wait(timeout=remaining)
    except subprocess.TimeoutExpired as error:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=10)
        raise TimeoutError(
            f'F0 operation deadline elapsed during {command[0]}'
        ) from error
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)

bootstrap_started = time.monotonic()
bootstrap_dir = Path('/kaggle/working/qwen3-vl-f0-r4-bootstrap')
bootstrap_dir.mkdir(parents=True, exist_ok=False)
bootstrap_result = {
    'started_at': datetime.now(UTC).isoformat(),
    'status': 'started',
}
try:
    input_root = Path('/kaggle/input')
    wheel_paths = list(input_root.rglob('vllm-0.28.0-cp38-abi3-manylinux_2_28_x86_64.whl'))
    lock_paths = [
        path
        for path in input_root.rglob('requirements.lock')
        if sha256_file(path) == 'd473b6e092e01f66313e12069ac8873f2ff1c6c3b84b51c1b37312229de5475c'
    ]
    if len(wheel_paths) != 1 or len(lock_paths) != 1:
        raise RuntimeError(
            'expected one pinned vLLM wheel and lockfile in attached inputs, '
            f'found wheels={wheel_paths}, locks={lock_paths}'
        )
    wheel_path = wheel_paths[0]
    lock_path = lock_paths[0]
    actual_wheel_sha256 = sha256_file(wheel_path)
    if actual_wheel_sha256 != 'addb0ffdaafd8155d75e9b3f5ddb3da28fdee9e8a7097ede91f7db2e9e1a3889':
        raise RuntimeError('mounted vLLM wheel does not match the pinned bytes')
    source_lock_lines = lock_path.read_text().splitlines()
    excluded_requirements = [
        line
        for line in source_lock_lines
        if line.startswith('sglang @ git+')
    ]
    if excluded_requirements != ['sglang @ git+https://github.com/sgl-project/sglang.git@8e04f66a7046fa083cda3f5b8e8325632a5af8fd#subdirectory=python']:
        raise RuntimeError(
            'shared lock SGLang exclusion changed: '
            f'{excluded_requirements}'
        )
    derived_lock_path = Path('/tmp/vllm-requirements.lock')
    derived_lock_path.write_text(
        '\n'.join(
            line
            for line in source_lock_lines
            if line != 'sglang @ git+https://github.com/sgl-project/sglang.git@8e04f66a7046fa083cda3f5b8e8325632a5af8fd#subdirectory=python'
        )
        + '\n'
    )
    derived_lock_sha256 = sha256_file(derived_lock_path)
    if derived_lock_sha256 != 'd07866e8a6961193a1285d1514560c46610413f9a05fc19738c286320a98c481':
        raise RuntimeError(
            'derived vLLM lock differs from the pinned transformation'
        )
    wheelhouse = wheel_path.parent
    runtime_provenance = Path('/tmp/vllm-runtime-provenance.json')
    runtime_provenance.write_text(json.dumps({
        'dataset_source': 'mahdiseddigh/kaggle-t4-stack-2026-08-28',
        'dataset_version': 1,
        'artifact_path': str(wheel_path),
        'artifact_sha256': actual_wheel_sha256,
        'lock_path': str(lock_path),
        'lock_sha256': 'd473b6e092e01f66313e12069ac8873f2ff1c6c3b84b51c1b37312229de5475c',
        'derived_lock_path': str(derived_lock_path),
        'derived_lock_sha256': derived_lock_sha256,
        'excluded_requirements': excluded_requirements,
    }, indent=2) + '\n')

    overrides_path = Path('/tmp/vllm-runtime-overrides.txt')
    overrides_path.write_text('flashinfer-python==0.6.17\n')
    uv_install_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--find-links',
        str(wheelhouse),
        'uv==0.12.7',
    ]
    with (bootstrap_dir / 'runtime-install.log').open('w') as install_log:
        run_bounded(uv_install_command, install_log)
    uv_executable = shutil.which('uv')
    if not uv_executable:
        raise RuntimeError('uv executable missing after offline installation')
    install_command = [
        uv_executable,
        'pip',
        'install',
        '--system',
        '--no-index',
        '--find-links',
        str(wheelhouse),
        '--override',
        str(overrides_path),
        '--upgrade',
        '--requirement',
        str(derived_lock_path),
    ]
    (bootstrap_dir / 'runtime-install-command.json').write_text(
        json.dumps({
            'uv_install': uv_install_command,
            'runtime_install': install_command,
            'overrides': overrides_path.read_text().splitlines(),
        }, indent=2) + '\n'
    )
    with (bootstrap_dir / 'runtime-install.log').open('a') as install_log:
        run_bounded(install_command, install_log)
    (bootstrap_dir / 'runtime-provenance.json').write_text(
        runtime_provenance.read_text()
    )
    with (bootstrap_dir / 'runtime-packages.json').open('w') as package_log:
        run_bounded(
            [sys.executable, '-m', 'pip', 'list', '--format', 'json'],
            package_log,
            merge_stderr=False,
        )
    bootstrap_result['status'] = 'passed'
except Exception as error:
    bootstrap_result |= {
        'status': 'failed',
        'error_type': type(error).__name__,
        'error': str(error),
    }
    raise
finally:
    bootstrap_result |= {
        'ended_at': datetime.now(UTC).isoformat(),
        'elapsed_s': round(time.monotonic() - bootstrap_started, 3),
    }
    (bootstrap_dir / 'bootstrap-result.json').write_text(
        json.dumps(bootstrap_result, indent=2) + '\n'
    )


In [ ]:
import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

output_dir = Path('/kaggle/working/qwen3-vl-f0-r4-probe')
probe_command = [
    sys.executable,
    '/tmp/PROTOTYPE-qwen3_vl_f0_probe.py',
    '--output-dir',
    str(output_dir),
    '--input-root',
    '/kaggle/input',
    '--runtime-provenance',
    '/tmp/vllm-runtime-provenance.json',
    '--startup-timeout-s',
    '1200',
    '--request-timeout-s',
    '180',
]
remaining = F0_OPERATION_DEADLINE - time.monotonic()
if remaining <= 0:
    raise TimeoutError('F0 operation deadline elapsed before probe launch')
probe_process = subprocess.Popen(probe_command, start_new_session=True)
Path('/kaggle/working/qwen3-vl-f0-r4-outer-process.json').write_text(
    json.dumps({'pid': probe_process.pid, 'pgid': probe_process.pid}, indent=2) + '\n'
)
try:
    returncode = probe_process.wait(timeout=remaining)
except subprocess.TimeoutExpired:
    outer_timeout = {
        'status': 'operation_deadline_elapsed',
        'soft_deadline_s': 1110,
        'hard_deadline_s': 1200,
        'probe_pid': probe_process.pid,
    }
    os.kill(probe_process.pid, signal.SIGINT)
    finalization_remaining = max(1, F0_HARD_DEADLINE - time.monotonic())
    try:
        returncode = probe_process.wait(timeout=finalization_remaining)
    except subprocess.TimeoutExpired:
        outer_timeout['hard_deadline_elapsed'] = True
        server_process_path = output_dir / 'server-process.json'
        if server_process_path.is_file():
            server_process = json.loads(server_process_path.read_text())
            try:
                os.killpg(int(server_process['pgid']), signal.SIGKILL)
            except ProcessLookupError:
                pass
        try:
            os.killpg(probe_process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        returncode = probe_process.wait(timeout=10)
    Path('/kaggle/working/qwen3-vl-f0-r4-outer-timeout.json').write_text(
        json.dumps(outer_timeout, indent=2) + '\n'
    )
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, probe_command)
print((output_dir / 'result.json').read_text())
